## Set Variables

In [11]:
import logging
logger = logging.getLogger("detectree2")

# Name of site being used
# site_name='Danum'
site_name='LemonsAB'
# site_name='Sepilok'

# Name of dataset
dataset_name = site_name + '1'

# Root directory
root_dir = "./" + site_name

# Geotiff used for training and predictions
geotiff_path = root_dir + "/" + site_name + ".tif"

# Geopackage with crowns used for training
crown_path = root_dir + "/" + site_name + ".gpkg"

# Directory tiles are stored
tiles_dir = root_dir + "/tiles"

# Directory of training data
train_location = tiles_dir + "/train"

# Directory of training output files
model_output_dir = root_dir + "/train_outputs"

# Directory of prediction tiles
pred_tiles_dir = root_dir + "/tiles_pred"

# Directory to store split geotiff
geotiff_split_output_dir = root_dir + "/tiles_split/"

## Tile

In [ ]:
from detectree2.preprocessing.tiling import tile_data, to_traintest_folders
import geopandas as gpd
import rasterio

# Read in crowns and match CRS to the image
data = rasterio.open(geotiff_path)
crowns = gpd.read_file(crown_path)
crowns = crowns.to_crs(data.crs.data)

# Set tiling parameters
buffer = 30
tile_width = 40
tile_height = 40
threshold = 0.6

# Tile the data for training
tile_data(geotiff_path, tiles_dir, buffer, tile_width, tile_height, crowns, threshold, mode="rgb")

# Create train/test folders
to_traintest_folders(tiles_dir, tiles_dir, test_frac=0.15)

## Inspect Tiles

In [ ]:
from detectron2.data import MetadataCatalog
from detectron2.utils.visualizer import Visualizer
from detectree2.models.train import combine_dicts
import cv2
from PIL import Image

dataset_dicts = combine_dicts(train_location, 1, "full") # The number gives the fold to visualise
trees_metadata = MetadataCatalog.get(dataset_name + "_train")

for d in dataset_dicts:
   img = cv2.imread(d["file_name"])
   visualizer = Visualizer(img[:, :, ::-1], metadata=trees_metadata, scale=0.3)
   out = visualizer.draw_dataset_dict(d)
   image = cv2.cvtColor(out.get_image()[:, :, ::-1], cv2.COLOR_BGR2RGB)
   display(Image.fromarray(image))

## Train

In [ ]:
from detectree2.models.train import register_train_data, MyTrainer, setup_cfg

register_train_data(train_location=train_location, name=dataset_name, val_fold=None)

# Set the base (pre-trained) model from the detectron2 model_zoo
base_model = "COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml"

trains = (dataset_name + "_full",) # Registered train data
tests = (dataset_name + "_full",)   # Registered validation data

cfg = setup_cfg(base_model=base_model, trains=trains, tests=tests, workers=4, eval_period=100, max_iter=3000, out_dir=model_output_dir)

trainer = MyTrainer(cfg=cfg, patience=5)
trainer.resume_or_load(resume=False)
trainer.train()

## Prediction

In [ ]:
from detectree2.preprocessing.tiling import tile_data
from detectree2.models.predict import predict_on_data
from detectree2.models.outputs import project_to_geojson, stitch_crowns, clean_crowns
from detectree2.models.train import setup_cfg
from detectron2.engine import DefaultPredictor

# Specify tiling parameters (should be similar to training)
buffer = 30
tile_width = 40
tile_height = 40
tile_data(geotiff_path, pred_tiles_dir, buffer, tile_width, tile_height)

trained_model = "./Models/urban_trees_Cambridge_20230630.pth"
cfg = setup_cfg(update_model=trained_model)
predictor = DefaultPredictor(cfg)
predict_on_data(pred_tiles_dir, pred_tiles_dir + "/predictions", predictor)

# Project tile predictions to geo-referenced crowns
project_to_geojson(pred_tiles_dir, pred_tiles_dir + "/predictions", pred_tiles_dir + "/predictions_geo")

# Stitch and clean crowns
crowns2 = stitch_crowns(pred_tiles_dir + "/predictions_geo")
clean = clean_crowns(crowns2, 0.6, confidence=0) # Filter low-confidence and overlapping crowns

# Simplify geometries for easier editing in GIS software
clean = clean.set_geometry(clean.simplify(0.3))

# Save to file
clean.to_file(root_dir + "/crowns_out.gpkg", driver="GPKG")